# Day 2, Session 2 -- Exercises: LangChain Development & Tool Integration

**Engineering context:** You are the engineer building LLM-powered applications with LangChain. Consultants are your users -- they need reusable analysis templates, composable pipelines, custom analytical tools, and knowledge retrieval systems they can query in natural language.

**Structure:**
- Exercise 1 (Core): Custom Consulting Tools -- build and bind tools to an LLM
- Exercise 2 (Core): LCEL Analysis Pipeline -- build a multi-step chain
- Exercise 3 (Core): Custom RAG with Metadata Filtering
- Optional Exercise 1 (Intermediate): Tool-Augmented Chain
- Optional Exercise 2 (Intermediate): Document Chunking Optimizer
- Optional Exercise 3 (Intermediate): Full RAG Pipeline with Tool Selection

In [ ]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
import os

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

## Recap

In the demos you learned how to:
1. Use ChatModels and PromptTemplates as LangChain's core building blocks (Demo 1)
2. Compose multi-step chains with LCEL's pipe operator (Demo 2)
3. Create custom tools with the `@tool` decorator and bind them to an LLM (Demo 3)
4. Split documents into overlapping chunks for retrieval (Demo 4)
5. Build a simple RAG pipeline combining retrieval + generation (Demo 5)

Now you will build your own implementations of each pattern.

---
## Exercise 1 (Easy): Custom Consulting Tools

**Reference:** Demo 3 (Custom Tools)

> **Hint:** The function-calling loop has 3 phases: (1) send message with tools, (2) detect and execute tool_calls, (3) send tool result back to the model.

**Goal:** Create three custom tools using the `@tool` decorator that mirror analytical utilities consultants use daily, then bind them to an LLM so it can decide which tool to call.

**What you will build:**
1. `competitor_analysis_tool(company, competitors)` -- Analyzes a company against its competitors
2. `market_sizing_estimate(total_population, serviceable_pct, obtainable_pct, avg_deal_value)` -- Calculates TAM/SAM/SOM
3. `engagement_roi_tool(engagement_cost, projected_savings, implementation_months)` -- Calculates ROI and payback

**Key reminders from Demo 3:**
- The `@tool` decorator turns a Python function into a LangChain tool
- The docstring becomes the tool description the LLM reads
- `.invoke()` lets you call tools directly for testing
- `.bind_tools()` attaches tool schemas to the LLM

### Step 1: Create `competitor_analysis_tool`

**Requirements:**
- Parameters: `company` (str), `competitors` (list[str])
- Returns a dict with:
  - `focal_company`: the company name
  - `competitors_analyzed`: count of competitors
  - `competitor_list`: the list of competitors
  - `dimensions`: list of analysis dimensions (e.g., ["Market Share", "Product Portfolio", "Geographic Reach", "Innovation Pipeline"])
  - `methodology`: a string like "McKinsey Competitive Positioning Framework"
  - `recommendation`: a string suggesting the primary competitive threat

**Hint:** Look at how `market_sizing_tool` was structured in Demo 3a -- same pattern, different logic.

In [ ]:
# Exercise 1, Step 1: Create competitor_analysis_tool (provided as example)

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import Document

@tool
def competitor_analysis_tool(company: str, competitors: str) -> str:
    """Analyze a focal company against a list of competitors.
    Returns competitive positioning insights including market share,
    strengths, and strategic threats."""
    return f"Competitive analysis for {company} vs {competitors}: {company} holds ~25% market share. Key differentiators include brand strength and operational efficiency. Primary threats: pricing pressure from {competitors}."

# TODO: Create market_sizing_estimate tool using @tool decorator
# Parameters: total_population (int), serviceable_pct (float), obtainable_pct (float), avg_deal_value (float)
# Hint: Calculate TAM = total_population * avg_deal_value
#        SAM = TAM * serviceable_pct/100, SOM = SAM * obtainable_pct/100
#        Return a formatted string with all three values

# @tool
# def market_sizing_estimate(...) -> str:
#     """Calculate TAM/SAM/SOM market sizing..."""
#     ...


# TODO: Create engagement_roi_tool using @tool decorator
# Parameters: engagement_cost (float), annual_savings (float), years (int)
# Hint: total_benefit = annual_savings * years, roi_pct = ((total_benefit - engagement_cost) / engagement_cost) * 100
#        Return a formatted string with ROI, payback period, etc.

# @tool
# def engagement_roi_tool(...) -> str:
#     """Calculate engagement ROI metrics..."""
#     ...

### Step 2: Create `market_sizing_estimate`

**Requirements:**
- Parameters: `total_population` (float), `serviceable_pct` (float, 0-1), `obtainable_pct` (float, 0-1), `avg_deal_value` (float)
- Calculations:
  - TAM = total_population * avg_deal_value
  - SAM = TAM * serviceable_pct
  - SOM = SAM * obtainable_pct
- Returns a dict with formatted dollar values for TAM, SAM, SOM plus methodology string

**Hint:** Use f-strings like `f"${tam:,.0f}"` to format large numbers as currency.

In [ ]:
# Exercise 1, Step 2: Create market_sizing_estimate

# TODO: Define the tool using @tool decorator
# Docstring should explain: "Calculate Total Addressable Market (TAM), 
# Serviceable Addressable Market (SAM), and Serviceable Obtainable Market (SOM)..."

# YOUR CODE HERE


### Step 3: Create `engagement_roi_tool`

**Requirements:**
- Parameters: `engagement_cost` (float), `projected_savings` (float), `implementation_months` (int)
- Calculations:
  - roi = ((projected_savings - engagement_cost) / engagement_cost) * 100
  - payback_months = engagement_cost / (projected_savings / 12)
- Returns a dict with:
  - `engagement_cost`: formatted as dollar string
  - `projected_annual_savings`: formatted as dollar string
  - `roi_pct`: formatted as percentage string (e.g., "650.0%")
  - `payback_period_months`: formatted to 1 decimal place
  - `implementation_timeline`: e.g., "6 months"
  - `verdict`: "Strong ROI" if roi > 200, "Moderate ROI" if roi > 100, else "Marginal ROI"

In [ ]:
# Exercise 1, Step 3: Create engagement_roi_tool

# TODO: Define the tool using @tool decorator
# Docstring should explain when this tool is useful (calculating consulting engagement ROI)

# YOUR CODE HERE


### Step 4: Test Tools Directly

Before binding to an LLM, verify each tool works by calling `.invoke()` directly.

In [ ]:
# Exercise 1, Step 4: Test each tool directly with .invoke()

# TODO: Uncomment and run these tests after implementing your tools

# Test 1: Competitor analysis
# result1 = competitor_analysis_tool.invoke({"company": "Tesla", "competitors": ["BYD", "Rivian", "Lucid"]})
# print(f"competitor_analysis_tool: {result1}")

# Test 2: Market sizing
# result2 = market_sizing_estimate.invoke({"total_population": 5_000_000, "serviceable_pct": 0.3, "obtainable_pct": 0.1, "avg_deal_value": 50_000})
# print(f"\nmarket_sizing_estimate: {result2}")

# Test 3: Engagement ROI
# result3 = engagement_roi_tool.invoke({"engagement_cost": 2_000_000, "projected_savings": 15_000_000, "implementation_months": 6})
# print(f"\nengagement_roi_tool: {result3}")

### Step 5: Bind Tools to LLM and Test Tool Selection

Bind all three tools to an LLM and verify it selects the correct tool for different queries.

In [ ]:
# Exercise 1, Step 5: Bind tools to LLM and test
# Uncomment and run once you have all 3 tools defined.

# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)
# llm_with_tools = llm.bind_tools([competitor_analysis_tool, market_sizing_estimate, engagement_roi_tool])

# # Test -- each query should trigger a different tool
# queries = [
#     "Estimate the market size for premium electric SUVs with 10 million potential buyers, 25% serviceable, 8% obtainable, and $65,000 average price",
#     "Analyze Tesla's competitive position against BYD, Rivian, and Lucid",
#     "We spent $2M on a cost optimization engagement that saves $800K per year. What is the 5-year ROI?",
# ]
# for q in queries:
#     response = llm_with_tools.invoke(q)
#     print(f"Query: {q[:60]}...")
#     print(f"Tool calls: {response.tool_calls}")
#     print()

### Expected Output

When you invoke each tool directly and then bind them to an LLM, you should see output similar to:

In [ ]:
expected_output = """
competitor_analysis_tool: {'focal_company': 'Tesla', 'competitors_analyzed': 3,
  'competitor_list': ['BYD', 'Rivian', 'Lucid'],
  'dimensions': ['Market Share', 'Product Portfolio', 'Geographic Reach', 'Innovation Pipeline'],
  'methodology': 'McKinsey Competitive Positioning Framework',
  'recommendation': 'Conduct deep-dive on BYD as primary competitive threat to Tesla'}

market_sizing_estimate: {'TAM': '$250,000,000,000', 'SAM': '$75,000,000,000',
  'SOM': '$7,500,000,000', 'methodology': 'Top-down market sizing with TAM/SAM/SOM framework'}

engagement_roi_tool: {'engagement_cost': '$2,000,000', 'projected_annual_savings': '$15,000,000',
  'roi_pct': '650.0%', 'payback_period_months': '1.6',
  'implementation_timeline': '6 months', 'verdict': 'Strong ROI'}

LLM tool calls: [{'name': 'market_sizing_estimate',
  'args': {'total_population': 10000000, 'serviceable_pct': 0.25,
           'obtainable_pct': 0.08, 'avg_deal_value': 65000},
  'id': '...', 'type': 'tool_call'}]
"""
print(expected_output)

---
## Exercise 2 (Easy): LCEL Analysis Pipeline

**Reference:** Demo 2 (LCEL Chains)

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Goal:** Build a 3-step LCEL pipeline that processes market data through analysis, extraction, and summarization stages -- mirroring how a consulting team processes raw research into client-ready insights.

**What you will build:**
1. **Step 1 (Analyze):** Take raw market data and produce a detailed analysis
2. **Step 2 (Extract):** Parse the analysis into structured JSON with key metrics
3. **Step 3 (Summarize):** Generate a concise executive summary from the structured data

**Key reminders from Demo 2:**
- Chain pattern: `prompt | llm | parser`
- Use `StrOutputParser()` for text output, `JsonOutputParser()` for structured output
- Each chain is independent -- pass output of one as input to the next

### Input Data

Here is the raw market data your pipeline will process:

In [ ]:
# Exercise 2: Input data for the pipeline

market_data = """Q4 2024 Cloud Infrastructure Market Report:
- Total market size: $84.2B (up from $65.3B in Q4 2023)
- AWS market share: 31% ($26.1B revenue)
- Azure market share: 25% ($21.1B revenue) 
- Google Cloud market share: 12% ($10.1B revenue)
- Year-over-year growth: 28.9%
- Enterprise adoption rate: 94% of Fortune 500 companies use multi-cloud
- Key trend: AI workloads now represent 35% of new cloud spending
- Margin pressure: Average gross margins declined 2pp to 62% due to AI infrastructure costs
- Forecast: Market expected to reach $110B by Q4 2025"""

print("Market data loaded. Build your pipeline below.")

### Step 1: Analysis Chain

Build a chain that takes raw market data and produces a detailed competitive analysis.

**Requirements:**
- System prompt should instruct the LLM to act as a strategy consultant
- Human prompt should pass in `{market_data}` and ask for competitive analysis
- Output: plain text (use `StrOutputParser()`)

In [ ]:
# Exercise 2, Step 1: Build the analysis chain

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3"))
)

# TODO: Create the analysis prompt template
# Hint: Fill in the system message below
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strategy consultant analyzing market data. Provide competitive insights, identify key trends, and highlight strategic implications."),
    ("human", "Analyze the following market data and provide competitive insights:\n\n{market_data}")
])

# Chain: prompt -> llm -> parse text output (provided)
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Test with the market data from the data cell above
# analysis_result = analysis_chain.invoke({"market_data": market_data})
# print("Step 1 - Analysis Result:")
# print(analysis_result[:500])

### Step 2: JSON Extraction Chain

Build a chain that takes the analysis text and extracts structured metrics.

**Requirements:**
- System prompt should instruct JSON output with specific keys:
  - `market_size_b` (number)
  - `yoy_growth_pct` (number)
  - `market_leader` (string)
  - `key_trend` (string)
  - `risk_factors` (list of strings)
  - `opportunity_score` (1-10)
- Use `JsonOutputParser()`

**Hint from Demo 2b:** Tell the LLM to output ONLY valid JSON, no markdown fences.

In [ ]:
# Exercise 2, Step 2: Build the JSON extraction chain

# TODO: Create a prompt that instructs the LLM to extract key metrics as JSON
# Hint: The prompt should ask for keys like market_leader, market_share_pct,
#       growth_rate, key_risk, key_opportunity
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the key metrics from the following analysis as a JSON object. Include: market_leader, market_share_pct, growth_rate, key_risk, key_opportunity."),
    ("human", "{analysis_text}")
])

# Chain: prompt -> llm -> parse JSON output (provided)
extraction_chain = extraction_prompt | llm | JsonOutputParser()

# Test with result from Step 1 (uncomment after Step 1 works)
# metrics = extraction_chain.invoke({"analysis_text": analysis_result})
# print("Step 2 - Extracted Metrics:")
# print(json.dumps(metrics, indent=2))

### Step 3: Executive Summary Chain

Build a chain that takes the structured metrics and generates a 3-sentence executive summary.

**Requirements:**
- Accept `{metrics_json}` as input (you will pass `json.dumps(metrics)` from Step 2)
- Output should be exactly 3 sentences suitable for a C-suite audience
- Use `StrOutputParser()`

In [ ]:
# Exercise 2, Step 3: Build the executive summary chain

# TODO: Create a prompt for executive summary generation
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior consultant writing executive summaries. Produce a concise 3-sentence summary from the metrics provided. Focus on the most important takeaway, the key risk, and the recommended action."),
    ("human", "Produce a 3-sentence executive summary from these metrics:\n{metrics_json}")
])

# Chain (provided)
summary_chain = summary_prompt | llm | StrOutputParser()

# Test with metrics from Step 2 (uncomment after Step 2 works)
# exec_summary = summary_chain.invoke({"metrics_json": json.dumps(metrics)})
# print("Step 3 - Executive Summary:")
# print(exec_summary)

### Full Pipeline Run

Once all three steps work individually, run the complete pipeline end-to-end:

In [ ]:
# Exercise 2: Full pipeline (uncomment after implementing all steps)

# print("="*60)
# print("FULL PIPELINE: Market Data -> Analysis -> Metrics -> Summary")
# print("="*60)
#
# # Step 1: Analyze
# analysis_result = analysis_chain.invoke({"market_data": market_data})
# print(f"\n[Step 1] Analysis length: {len(analysis_result)} chars")
#
# # Step 2: Extract metrics
# metrics = extraction_chain.invoke({"analysis_text": analysis_result})
# print(f"[Step 2] Extracted {len(metrics)} metrics")
# print(f"  Market leader: {metrics.get('market_leader')}")
# print(f"  YoY Growth: {metrics.get('yoy_growth_pct')}%")
# print(f"  Opportunity Score: {metrics.get('opportunity_score')}/10")
#
# # Step 3: Executive summary
# exec_summary = summary_chain.invoke({"metrics_json": json.dumps(metrics)})
# print(f"\n[Step 3] Executive Summary:")
# print(f"  {exec_summary}")

---
## Exercise 3 (Easy): Custom RAG with Metadata Filtering

**Reference:** Demo 4 (Document Splitting) + Demo 5 (RAG Pipeline)

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Goal:** Build a RAG pipeline that filters documents by metadata (practice area) before retrieval. This simulates a real consulting knowledge management system where you search within a specific practice area.

**What you will build:**
1. A knowledge base of documents with `practice_area` metadata
2. A retrieval function that first filters by practice area, then searches by keywords
3. A RAG chain that answers questions scoped to a specific practice

### Step 1: Create the Knowledge Base

Create at least 6 `Document` objects across 3 practice areas. Each document should have metadata including `practice_area`, `source`, and `year`.

In [ ]:
# Exercise 3, Step 1: Create knowledge base with metadata (provided)

knowledge_base = [
    # Strategy practice
    Document(page_content="Market entry frameworks should evaluate market attractiveness (size, growth, profitability) and competitive intensity (number of players, barriers to entry, substitution threat). Use Porter's Five Forces for systematic assessment.",
             metadata={"practice_area": "strategy", "source": "Strategy Practice Guide", "year": 2024}),
    Document(page_content="Competitive positioning analysis maps players on value vs. cost dimensions. Identify white space opportunities where no competitor has a strong presence. First-mover advantage matters most in winner-take-all markets.",
             metadata={"practice_area": "strategy", "source": "Competitive Intelligence Handbook", "year": 2024}),

    # Operations practice
    Document(page_content="Supply chain optimization starts with a total-cost-to-serve analysis. Map all cost drivers from raw material to customer delivery. Typical savings range from 10-25% through network redesign, procurement consolidation, and demand planning improvements.",
             metadata={"practice_area": "operations", "source": "Operations Excellence Guide", "year": 2024}),
    Document(page_content="Lean manufacturing principles focus on eliminating waste in seven categories: overproduction, waiting, transport, overprocessing, inventory, motion, and defects. Start with value stream mapping to identify the biggest waste sources.",
             metadata={"practice_area": "operations", "source": "Lean Operations Manual", "year": 2023}),

    # Digital practice
    Document(page_content="Cloud migration follows a 6-R framework: Rehost, Replatform, Repurchase, Refactor, Retire, Retain. Start with an application portfolio assessment to categorize each workload. Typical enterprise migrations take 12-24 months.",
             metadata={"practice_area": "digital", "source": "Digital Transformation Playbook", "year": 2024}),
    Document(page_content="AI/ML implementation requires a phased approach: start with data readiness assessment, then pilot with high-impact/low-risk use cases, then scale. Most failed AI projects fail due to data quality issues, not model complexity.",
             metadata={"practice_area": "digital", "source": "AI Strategy Guide", "year": 2024}),
]

print(f"Knowledge base: {len(knowledge_base)} documents across 3 practice areas")

### Step 2: Build Metadata-Filtered Retrieval

Create a retrieval function that:
1. Filters documents by `practice_area` metadata
2. Then performs keyword search within the filtered set
3. Returns top-k results

**Hint from Demo 5a:** Use the `simple_retrieve` pattern but add a filtering step first.

In [ ]:
# Exercise 3, Step 2: Metadata-filtered retrieval function (mostly provided)

def filtered_retrieve(query, practice_area, k=2):
    """Retrieve top-k documents matching the practice area."""
    # Step 1: Filter by practice area
    filtered = [doc for doc in knowledge_base if doc.metadata["practice_area"] == practice_area]

    # Step 2: Score by keyword overlap (simple but effective)
    query_words = set(query.lower().split())
    scored = []
    for doc in filtered:
        doc_words = set(doc.page_content.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))

    # Step 3: Sort by score descending and return top-k
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k]]

# Test
results = filtered_retrieve("supply chain optimization cost reduction", "operations")
for doc in results:
    print(f"  [{doc.metadata['practice_area']}] {doc.page_content[:80]}...")

### Step 3: Build the RAG Chain

Create an LCEL chain that takes retrieved context and a question, then generates a grounded answer.

**Requirements:**
- System prompt should instruct the LLM to answer ONLY from the provided context
- Include the practice area in the system prompt for additional grounding
- If the answer is not in the context, say so explicitly

In [ ]:
# Exercise 3, Step 3: Build RAG chain with metadata filtering

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

# RAG prompt template (provided)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledge assistant for the {practice_area} practice at McKinsey. "
               "Answer the question using ONLY the context provided. Cite your sources.\n\n"
               "Context:\n{context}"),
    ("human", "{question}")
])

# RAG chain (provided)
rag_chain = rag_prompt | llm | StrOutputParser()

# TODO: Test with a query. Steps:
# 1. Call filtered_retrieve to get relevant docs
# 2. Format the docs as context text
# 3. Invoke the chain with practice_area, context, and question
#
# Example:
# query = "What frameworks should we use for market entry analysis?"
# docs = filtered_retrieve(query, "strategy", k=2)
# context = "\n\n".join([d.page_content for d in docs])
# answer = rag_chain.invoke({"practice_area": "strategy", "context": context, "question": query})
# print(answer)

### Test: Cross-Practice Queries

Verify that your metadata filtering works correctly by asking a strategy question against the operations practice -- it should return "not covered" or irrelevant results.

In [ ]:
# Exercise 3: Cross-practice test (uncomment after implementing)

# # This should work -- strategy question to strategy practice
# q1 = "What frameworks are used for market entry?"
# retrieved1 = filtered_retrieve(q1, "strategy")
# context1 = "\n".join(f"- {doc.page_content}" for doc in retrieved1)
# answer1 = rag_chain.invoke({"practice_area": "strategy", "context": context1, "question": q1})
# print(f"[strategy] Q: {q1}")
# print(f"A: {answer1}\n")
#
# # This should fail gracefully -- strategy question to operations practice
# retrieved2 = filtered_retrieve(q1, "operations")
# context2 = "\n".join(f"- {doc.page_content}" for doc in retrieved2)
# answer2 = rag_chain.invoke({"practice_area": "operations", "context": context2, "question": q1})
# print(f"[operations] Q: {q1}")
# print(f"A: {answer2}")

---
---
## Optional Exercises


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
These exercises combine multiple demo concepts and are at an intermediate level. Attempt them if you finish the core exercises early.

---
## Optional Exercise 1 (Intermediate): Tool-Augmented Chain

**Reference:** Demo 2 (LCEL Chains) + Demo 3 (Custom Tools)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Goal:** Build an LCEL chain that uses custom tools to answer questions. The chain should:
1. Receive a user question
2. Use an LLM with bound tools to decide which tool to call
3. Execute the selected tool
4. Pass the tool result back to the LLM for a final answer

This combines the chain composition from Demo 2 with the tool binding from Demo 3.

In [ ]:
# Optional Exercise 1: Tool-Augmented Chain

# TODO: Step 1 - Define 2-3 tools (reuse from Exercise 1 or create new ones)

# TODO: Step 2 - Create an LLM with tools bound
# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)
# llm_with_tools = llm.bind_tools([your_tools_here])

# TODO: Step 3 - Build the chain that:
#   a) Invokes the LLM with tools to get tool_calls
#   b) Executes the selected tool
#   c) Passes the result back to the LLM for a human-readable answer
#
# Hint: You'll need to manually orchestrate this since basic LCEL doesn't
# handle tool execution automatically. Use a function like:
#
# def answer_with_tools(question):
#     # Step A: Ask LLM which tool to use
#     response = llm_with_tools.invoke(question)
#     
#     # Step B: Execute the tool
#     if response.tool_calls:
#         tool_call = response.tool_calls[0]
#         tool_name = tool_call["name"]
#         tool_args = tool_call["args"]
#         
#         # Map tool names to functions
#         tool_map = {"market_sizing_estimate": market_sizing_estimate, ...}
#         tool_result = tool_map[tool_name].invoke(tool_args)
#         
#         # Step C: Generate final answer
#         final_prompt = ChatPromptTemplate.from_template(
#             "The user asked: {question}\nTool '{tool_name}' returned: {tool_result}\nProvide a clear answer."
#         )
#         final_chain = final_prompt | llm | StrOutputParser()
#         return final_chain.invoke({...})
#     else:
#         return response.content

# TODO: Test with questions that trigger different tools
# print(answer_with_tools("What's the market size for 50M users at 5% adoption and $100 ARPU?"))

# YOUR CODE HERE


---
## Optional Exercise 2 (Intermediate): Document Chunking Optimizer

**Reference:** Demo 4 (Document Splitting)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Goal:** Test different `chunk_size` and `chunk_overlap` values and measure how they affect retrieval quality. You will:
1. Create a test document with known facts
2. Split it with different parameters
3. Measure which configurations retrieve the correct chunks for test queries

This helps build intuition for tuning chunking parameters in production RAG systems.

In [ ]:
# Optional Exercise 2: Document Chunking Optimizer

# Step 1: Create a test document with clearly separated facts
test_document = Document(
    page_content="""Section 1: Market Overview
The global AI market reached $196 billion in 2024. Growth is driven by enterprise adoption.
North America accounts for 42% of the market. Europe follows at 25%.

Section 2: Key Players
Microsoft leads in enterprise AI with Azure OpenAI Service. Google dominates in AI research publications.
Amazon's AWS offers the broadest ML service portfolio. Meta leads in open-source model releases.

Section 3: Technology Trends
Transformer architectures remain dominant for language tasks. Diffusion models lead in image generation.
Multimodal models combining text, image, and code are the fastest-growing segment.
Small language models (SLMs) are gaining traction for edge deployment.

Section 4: Investment Landscape
VC funding for AI startups reached $50 billion in 2024. Generative AI captured 40% of all AI funding.
Enterprise AI spending grew 35% year-over-year. The median Series A for AI companies is $12 million.""",
    metadata={"source": "ai_market_report.pdf"}
)

# Step 2: Define test queries with expected answers
test_queries = [
    {"query": "What is the global AI market size?", "expected_keyword": "$196 billion"},
    {"query": "Who leads in enterprise AI?", "expected_keyword": "Microsoft"},
    {"query": "What is the median Series A?", "expected_keyword": "$12 million"},
    {"query": "What are transformer architectures used for?", "expected_keyword": "language tasks"},
]

# TODO: Step 3 - Test different chunking configurations
# configurations = [
#     {"chunk_size": 100, "chunk_overlap": 0},
#     {"chunk_size": 100, "chunk_overlap": 20},
#     {"chunk_size": 200, "chunk_overlap": 30},
#     {"chunk_size": 200, "chunk_overlap": 50},
#     {"chunk_size": 300, "chunk_overlap": 50},
# ]
#
# for config in configurations:
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=config["chunk_size"],
#         chunk_overlap=config["chunk_overlap"]
#     )
#     chunks = splitter.split_documents([test_document])
#     
#     # Test retrieval for each query
#     hits = 0
#     for test in test_queries:
#         # Simple keyword retrieval
#         query_words = set(test["query"].lower().split())
#         best_chunk = max(chunks, key=lambda c: len(query_words & set(c.page_content.lower().split())))
#         if test["expected_keyword"].lower() in best_chunk.page_content.lower():
#             hits += 1
#     
#     accuracy = hits / len(test_queries) * 100
#     print(f"  chunk_size={config['chunk_size']:3d}, overlap={config['chunk_overlap']:2d} | "
#           f"{len(chunks)} chunks | Retrieval accuracy: {accuracy:.0f}%")

# YOUR CODE HERE


---
## Optional Exercise 3 (Intermediate): Full RAG Pipeline with Tool Selection

**Reference:** Demo 3 (Custom Tools) + Demo 5 (RAG Pipeline)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Goal:** Build a system where the LLM first decides whether to:
- Use a **calculation tool** (for quantitative questions), OR
- **Search the knowledge base** (for qualitative/factual questions)

Then generates an answer using the result. This is a simplified version of what agent frameworks do automatically.

**Architecture:**
```
User Question
    |
    v
[Router LLM] --> decides: tool OR knowledge_base
    |                           |
    v                           v
[Execute Tool]          [Retrieve from KB]
    |                           |
    v                           v
[Generate Final Answer with context]
```

In [ ]:
# Optional Exercise 3: Full RAG Pipeline with Tool Selection

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

# Step 1: Define a calculation tool
@tool
def calculate_cagr(start_value: float, end_value: float, years: int) -> dict:
    """Calculate Compound Annual Growth Rate (CAGR) given start value, end value, and number of years."""
    cagr = (end_value / start_value) ** (1 / years) - 1
    return {
        "cagr_pct": f"{cagr * 100:.1f}%",
        "start_value": start_value,
        "end_value": end_value,
        "years": years,
        "interpretation": f"The value grew at {cagr*100:.1f}% annually over {years} years"
    }

# Step 2: Knowledge base
kb = [
    "The consulting industry is valued at $300B globally as of 2024, with management consulting representing approximately 40% of the total.",
    "McKinsey, BCG, and Bain (MBB) collectively hold approximately 10% market share of the global consulting market by revenue.",
    "Digital transformation consulting is the fastest-growing segment, expanding at 25% CAGR since 2020.",
    "The average utilization rate for top-tier consulting firms is 60-70%, with partners typically at 40-50%.",
    "Consulting fee structures include: time & materials (hourly), fixed-fee projects, success-based fees, and retainer arrangements.",
]

# TODO: Step 3 - Build a router that decides: "tool" or "knowledge_base"
# Hint: Use the LLM with a system prompt that says:
#   "You are a router. Given a user question, decide if it requires:
#    1. A calculation (output 'TOOL') 
#    2. Knowledge lookup (output 'KB')
#    Output ONLY 'TOOL' or 'KB', nothing else."
#
# router_prompt = ChatPromptTemplate.from_messages([...])
# router_chain = router_prompt | llm | StrOutputParser()

# TODO: Step 4 - Build the full pipeline function
# def answer_question(question):
#     # Route the question
#     route = router_chain.invoke({"question": question}).strip()
#     
#     if route == "TOOL":
#         # Use tool
#         llm_with_tools = llm.bind_tools([calculate_cagr])
#         tool_response = llm_with_tools.invoke(question)
#         if tool_response.tool_calls:
#             result = calculate_cagr.invoke(tool_response.tool_calls[0]["args"])
#             context = f"Tool result: {result}"
#         else:
#             context = "No tool was called."
#     else:
#         # Search knowledge base
#         query_words = set(question.lower().split())
#         scored = [(len(query_words & set(doc.lower().split())), doc) for doc in kb]
#         scored.sort(key=lambda x: x[0], reverse=True)
#         context = "\n".join(f"- {doc}" for _, doc in scored[:2])
#     
#     # Generate final answer
#     final_prompt = ChatPromptTemplate.from_messages([
#         ("system", "Answer the question based on this context:\n{context}"),
#         ("human", "{question}")
#     ])
#     final_chain = final_prompt | llm | StrOutputParser()
#     return final_chain.invoke({"context": context, "question": question})

# TODO: Step 5 - Test with different question types
# test_questions = [
#     "What is the size of the consulting industry?",  # Should route to KB
#     "Calculate the CAGR if revenue grew from $50M to $120M over 5 years",  # Should route to TOOL
#     "What is the average utilization rate for consulting firms?",  # Should route to KB
#     "If a market was $10B in 2020 and $25B in 2024, what is the CAGR?",  # Should route to TOOL
# ]
#
# for q in test_questions:
#     route = router_chain.invoke({"question": q}).strip()
#     answer = answer_question(q)
#     print(f"[{route}] Q: {q}")
#     print(f"     A: {answer}\n")

# YOUR CODE HERE


---
## Summary

In this exercise notebook you practiced:

1. **Exercise 1:** Creating custom tools with `@tool` and binding them to an LLM for tool selection
2. **Exercise 2:** Building multi-step LCEL pipelines with different output parsers
3. **Exercise 3:** Implementing metadata-filtered RAG for practice-scoped knowledge retrieval
4. **Optional 1:** Orchestrating tool execution within a chain (manual agent loop)
5. **Optional 2:** Tuning chunking parameters and measuring retrieval quality
6. **Optional 3:** Building a router that decides between tools and knowledge base search

**Key takeaways:**
- LangChain's `@tool` decorator + `bind_tools()` lets LLMs decide WHICH tool to use, but YOUR code executes it
- LCEL chains are composable but independent -- you manually pass data between them (LangGraph automates this)
- RAG quality depends on retrieval quality -- metadata filtering, chunking strategy, and (in Day 3) embeddings all matter

**Next:** Session 3 introduces LangGraph, which automates the state passing and tool execution you did manually here.